# F26-13 — Phase 3 on Colab: baseline, CASP and the control experiments

T4 GPU, *Runtime → Run all*. Sections 1–4 rebuild the environment and the data (same as Phase 2);
section 5 trains the detector (Member 2, ~30 min, skip if `best.pt` is already in Drive); section 6 builds the
detection cache; section 7 runs the validation grid (Member 3); sections 8–9 run the experiments and build the tables.
Everything that must survive the session is copied to `MyDrive/F26-13/`.
Decisions: DL-30 … DL-37 in `docs/decision_log.md`.

## 1. GPU, Drive, code

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/proj
!git clone https://github.com/kashishfatima999/Confidence-Aware-Self-Prompting-for-Robust-Polyp-Segmentation.git /content/proj
%cd /content/proj
# !git checkout kashish-phase3   # <- uncomment while the Phase 3 branch is not merged into main yet
!git log -1 --format="Our commit: %h %s"
!git clone -q https://github.com/sajjad-sh33/YOLO_SAM2.git external/YOLO_SAM2

## 2. Packages (DL-26: Colab's own numpy/Pillow/OpenCV; only the model packages are installed) and tests

In [ ]:
!pip install -q -r requirements-colab.txt
!SAM2_BUILD_CUDA=0 pip install -q "git+https://github.com/facebookresearch/sam2.git@2b90b9f5ceec907a1c18123530e92e794ad901a4"
!pip freeze | grep -i -E "^torch==|^ultralytics==|^numpy==|^pillow==|SAM-2"
!python -m unittest discover -s tests 2>&1 | tail -3

## 3. Weights (Drive, downloaded once)

In [ ]:
import os, shutil
W = "/content/drive/MyDrive/F26-13/weights"
os.makedirs(W, exist_ok=True); os.makedirs("weights", exist_ok=True)
!wget -q -nc -P {W} https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt
!cd {W} && python -c "from ultralytics import YOLO; YOLO('yolov8m.pt')"
!ln -sf {W}/sam2_hiera_large.pt weights/sam2_hiera_large.pt
!ls -la {W}

## 4. Data (same as Phase 2: Kaggle copies, our prepare scripts, committed split)

In [ ]:
import os, glob, json, shutil
import kagglehub
import pandas as pd
from PIL import Image
os.chdir("/content/proj")

p = kagglehub.dataset_download("debeshjha1/kvasirseg")
src = f"{p}/Kvasir-SEG/Kvasir-SEG"
dest = "data/raw/kvasir-seg/Kvasir-SEG"

shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(f"{src}/images", f"{dest}/images")   # copy, never move
shutil.copytree(f"{src}/masks", f"{dest}/masks")

boxes = {}
for csv_path in sorted(glob.glob(f"{src}/bbox/*.csv")):
    image_id = os.path.splitext(os.path.basename(csv_path))[0]
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    w, h = Image.open(f"{dest}/images/{image_id}.jpg").size
    boxes[image_id] = {"height": h, "width": w,
                       "bbox": [{"label": "polyp", "xmin": int(r["xmin"]), "ymin": int(r["ymin"]),
                                 "xmax": int(r["xmax"]), "ymax": int(r["ymax"])} for _, r in df.iterrows()]}
with open(f"{dest}/kavsir_bboxes.json", "w") as f:
    json.dump(boxes, f)
print("Box entries written:", len(boxes))
!ls {dest}; ls {dest}/images | wc -l; ls {dest}/masks | wc -l

In [ ]:
!python src/data/prepare_kvasir.py | head -3
p2 = kagglehub.dataset_download("balraj98/cvcclinicdb")
shutil.rmtree("data/raw/cvc-clinicdb", ignore_errors=True)
shutil.copytree(f"{p2}/PNG", "data/raw/cvc-clinicdb/PNG")
!python src/data/prepare_clinicdb.py | head -2
!python src/data/make_corruptions.py | tail -1

## 5. Train the detector (Member 2, DL-31)

Recipe: `configs/yolo_train.yaml`. About 30 minutes on a T4. `best.pt` and `summary.json` are copied to Drive;
if they are already there, this cell reuses them instead of training again.

In [ ]:
import json, os, shutil
YOLO_DIR = "/content/drive/MyDrive/F26-13/yolo/yolov8m_kvasir_s42"
if os.path.exists(f"{YOLO_DIR}/weights/best.pt"):
    print("reusing trained weights from Drive")
    shutil.copytree(YOLO_DIR, "runs/yolo/yolov8m_kvasir_s42", dirs_exist_ok=True)
else:
    !python src/train_yolo.py
    shutil.copytree("runs/yolo/yolov8m_kvasir_s42", YOLO_DIR, dirs_exist_ok=True)
print(json.dumps(json.load(open("runs/yolo/yolov8m_kvasir_s42/summary.json")), indent=2))
BEST = "runs/yolo/yolov8m_kvasir_s42/weights/best.pt"

## 6. Detection cache (Member 2, DL-33): YOLO once per dataset at confidence 0.05

In [ ]:
DET_DIR = "/content/drive/MyDrive/F26-13/detections"
os.makedirs(DET_DIR, exist_ok=True)
for ds in ["kvasir_val", "kvasir_test", "clinicdb"]:
    !python src/pipeline/detect.py --weights {BEST} --dataset {ds}
!cp runs/detections/*.json {DET_DIR}/
!ls -la {DET_DIR}

In [ ]:
# Recall vs confidence threshold on the validation split, straight from the cache (report figure 2.2)
import sys, json, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "src/data"); sys.path.insert(0, "src/eval")
import common, metrics
cache = json.load(open("runs/detections/kvasir_val.json"))["images"]
ths = np.linspace(0.05, 0.95, 19); recalls, fps = [], []
for th in ths:
    tp = fp = fn = 0
    for image_id, e in cache.items():
        gt = common.mask_to_boxes(common.load_binary_mask(f"data/processed/kvasir/masks/val/{image_id}.png"))
        pred = [tuple(b[:4]) for b in sorted(e["boxes"], key=lambda b: -b[4]) if b[4] >= th]
        a, b, c = metrics.match_boxes(pred, gt); tp, fp, fn = tp + a, fp + b, fn + c
    recalls.append(tp / (tp + fn)); fps.append(fp)
plt.plot(ths, recalls, marker="o", label="polyp recall (IoU>=0.5)"); plt.xlabel("confidence threshold"); plt.ylabel("recall")
plt.twinx().plot(ths, fps, marker="s", color="C1", label="false-positive boxes"); plt.title("kvasir_val: recall and false positives vs threshold")
plt.savefig("docs/figures/phase3_recall_vs_threshold.png", dpi=120, bbox_inches="tight"); plt.show()
print("recall at 0.5:", round(recalls[9], 3), "| at 0.25:", round(recalls[4], 3), "| at 0.05:", round(recalls[0], 3))

## 7. Validation grid for CASP (Member 3, DL-34) — `kvasir_val` only

27 runs from the cache (each ≈ 100 images × a few prompts). The best setting by mean Dice is written into the
CASP configs and into DL-34 **before** any test run.

In [ ]:
import itertools, json, pandas as pd
rows = []
for t_low, t_high, s_min in itertools.product([0.15, 0.25, 0.35], [0.6, 0.7, 0.8], [0.6, 0.7, 0.8]):
    name = f"grid/tl{t_low}_th{t_high}_s{s_min}"
    !python src/pipeline/run_experiment.py --config configs/experiments/casp.yaml --dataset kvasir_val --name {name} --set t_low={t_low} t_high={t_high} s_min={s_min} > /dev/null
    s = json.load(open(f"runs/{name}/kvasir_val/summary.json"))
    rows.append({"t_low": t_low, "t_high": t_high, "s_min": s_min, "mean_dice": s["mean_dice"], "mean_iou": s["mean_iou"],
                 "missed": s["images_with_empty_prediction"], "fp": s["detection_fp"], "prompts_per_image": s["prompts_per_image"]})
grid = pd.DataFrame(rows).sort_values("mean_dice", ascending=False)
os.makedirs("docs/tables/phase3", exist_ok=True); grid.to_csv("docs/tables/phase3/casp_val_grid.csv", index=False)
print(grid.head(10).to_string(index=False))
best = grid.iloc[0]; print("\nchosen:", dict(best[["t_low", "t_high", "s_min"]]))

In [ ]:
# Baseline on the validation split for reference (same images as the grid)
!python src/pipeline/run_experiment.py --config configs/experiments/baseline.yaml --dataset kvasir_val --name grid/baseline | tail -12

## 8. Experiments on the test sets (DL-35)

**Before running this cell**, write the chosen `t_low / t_high / s_min` into `configs/experiments/casp.yaml`,
`casp_all.yaml`, `casp_noiter.yaml` and `threshold_only.yaml` (t_low only) and into DL-34, commit, and pull here.
Until then the cell uses the `--set` overrides below so the run is still correct.

In [ ]:
T_LOW, T_HIGH, S_MIN = float(best["t_low"]), float(best["t_high"]), float(best["s_min"])   # from section 7
RUNS = {
    "baseline":       "--config configs/experiments/baseline.yaml",
    "threshold_only": f"--config configs/experiments/threshold_only.yaml --set t_low={T_LOW} t_high={T_LOW}",
    "casp":           f"--config configs/experiments/casp.yaml --set t_low={T_LOW} t_high={T_HIGH} s_min={S_MIN} --save-masks",
    "casp_all":       f"--config configs/experiments/casp_all.yaml --set t_low={T_LOW} t_high={T_HIGH} s_min={S_MIN}",
    "casp_noiter":    f"--config configs/experiments/casp_noiter.yaml --set t_low={T_LOW} t_high={T_HIGH} s_min={S_MIN}",
    "oracle":         "--config configs/experiments/oracle.yaml",
}
for ds in ["kvasir_test", "clinicdb"]:
    for name, args in RUNS.items():
        print(f"\n==== {name} on {ds}")
        !python src/pipeline/run_experiment.py {args} --dataset {ds} --name {name} | tail -13

In [ ]:
# Corrupted Kvasir test set: B and C only, if time remains (DL-35)
for setting in ["blur_s1", "blur_s2", "blur_s3", "brightness_contrast_s1", "brightness_contrast_s2", "brightness_contrast_s3", "noise_s1", "noise_s2", "noise_s3"]:
    !python src/pipeline/detect.py --weights {BEST} --dataset kvasir_corrupted/{setting} > /dev/null
    !python src/pipeline/run_experiment.py --config configs/experiments/baseline.yaml --dataset kvasir_corrupted/{setting} --name baseline | tail -1
    !python src/pipeline/run_experiment.py --config configs/experiments/casp.yaml --set t_low={T_LOW} t_high={T_HIGH} s_min={S_MIN} --dataset kvasir_corrupted/{setting} --name casp | tail -1
!cp runs/detections/*.json {DET_DIR}/

## 9. Tables and Drive backup

In [ ]:
!python src/eval/summarize.py
R = "/content/drive/MyDrive/F26-13/runs_phase3"
!mkdir -p {R}
!rsync -a --exclude 'masks' runs/ {R}/            # every per_image.csv, detections.jsonl, summary.json
!rsync -a runs/casp/kvasir_test/masks {R}/casp_kvasir_test_masks/ 2>/dev/null; rsync -a runs/baseline/kvasir_test/masks {R}/baseline_kvasir_test_masks/ 2>/dev/null
!cp docs/figures/phase3_*.png /content/drive/MyDrive/F26-13/figures/ 2>/dev/null
!ls {R}

## What to commit after this run
`docs/tables/phase3/results.csv`, `results.md`, `casp_val_grid.csv`, `docs/figures/phase3_*.png`, the chosen thresholds in the
four CASP/threshold configs and in DL-34, plus a change-log row. Never `runs/`, weights or masks.